In [ ]:
!pip3 install gurobipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 45.3 MB/s eta 0:00:00


# Problema del flusso a costo minimo
Dato un grafo $G = (V, E)$, con capacita' su gli archi pari ad $h_{ij}$ per ogni arco $(i, j) \in E$, domanda (positiva o negativa) pari a $b_i$ per ogni nodo $i \in V$, e costo unitario del flusso $c_{ij}$ per ogni arco $(i, j) \in E$ il problema del flusso a costo minimo consiste nel trovare un flusso che rispetti tutte le domande a costo minimo.

## Generazione dei dati

Cominciamo con il generare in modo casuale un grafo $G$ (per facilitare la riproduzione dei risultati e' consigliabile fissare il seed del generatore di numeri casuali utilizzati, ad esempio usando np.random.seed) con almeno 150 nodi.

Generiamo inoltre le capacita' per ogni arco, nel range tra 1 e 10, e le domande per ogni nodo, assicurandoci che almeno un nodo abbia domanda positiva (e che il totale della domanda positiva sia maggiore di 15) ed almeno un altro nodo abbia domanda negativa (e che il totale della domanda negativa sia pari a quello della domanda positiva). Inoltre generiamo i costi per ogni arco nel grafo come numeri tra 1 ed 10.

In [2]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [3]:
np.random.seed(67)
adiac = np.random.randint(low=0, high=2, size=(150, 150))
costi = np.random.randint(low=1, high=11, size=(150, 150))
capacita = np.random.randint(low=1, high=11, size=(150, 150))
domande = np.zeros(150)
domande[3] = -15
domande[148] = 15
np.fill_diagonal(adiac, 0)

capacita = capacita * adiac
costi = costi * adiac
print(np.sum(adiac))

11177


## Risolviamo il problema con Gurobi

In [ ]:
mod = gp.Model()

#le variabili sono tuple (i, j) tali che non sono zero nella matrice di adiacenza e si fa cosi
var_idxs = np.nonzero(adiac)
var_idxs = [(i, j) for i, j in zip(var_idxs[0], var_idxs[1])]

#creo un dizionario che associa alle variabili le loro capacita cosi posso darlo come upper bound
capacita_variabili = {idx: adiac[idx] for idx in var_idxs}
#stessa cosa con i costi
costi_variabili = {idx: costi[idx] for idx in var_idxs}

archi = mod.addVars(var_idxs, lb = 0, ub = capacita_variabili, obj=costi_variabili, name="arcs")

ct1 = mod.addConstrs((archi.sum(v, '*') - archi.sum('*', v) == domande[v] for v in range(150)))

mod.optimize()

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Core(TM) i5-6300U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

: 

# Esercizio Calzature



Un industria calzaturiera produce 3 diversi modelli di scarpe (A, B e C) in 5 stabilimenti dislocati nel territorio. Le calzature vengono vendute al prezzo di 30, 55 e 40 euro rispettivamente. Dovendo decidere il piano produttivo per il mese seguente, il gruppo industriale deve decidere quali materie prime acquistare e come gestire la produzione per massimizzare il proprio profitto.

Per l’acquisto di materie prime, l’azienda ha a disposizione 1000 euro e puo’ acquistare pelle al prezzo di 4.5 euro/mq, lacci al prezzo di 73 cent/mt e suole al prezzo di 7 euro/unita’.

In un mese gli stabilimenti posso produrre le seguenti quantita di scapre (in paia):

| Modello      | Stabilimento 1 | Stabilimento 2 | Stabilimento 3 | Stabilimento 4 | Stabilimento 5 |
| ----------- | ----------- | ----------- | ----------- | ----------- | ----------- |
| A | 4 | 12 | 3 | 9 | 7 |
| B | 2 | 7 | 3 | 14 | 5 |
| C | 8 | 3 | 5 | 2 | 9 |


La produzione di una calzatura richiede una suola per tutti I modelli di scapre, ed in oltre: 1mq di pelle e 50cm di lacci per il modello A, 0.7mq di pelle e 45cm di lacci per il modello B, 1.35mq di pelle e 90cm di lacci per il modello C.

Considerando che le materie prime acquistate posso essere distribuite agli stabilimenti produttivi in tagli da 10mq di pelle, 5mt di lacci e blocchi di 3 paia di suole. Si determini l’approvigionamento del gruppo industriale, la distribuzione delle materie prime e la produzione in paia dei singoli stabilimenti in modo tale da massimizzare il profitto del gruppo, asssumendo che tutta la produzione venga venduta e sapendo ulteriormente che:

1. Devono essere prodotte almeno 5 paia di scarpe di tipo A

2. La produzione in paia dello stabilimento 3 deve essere pari a quella dello stabilimento 1

3. Lo stabilimento 4 ha gia’ ricevuto una commessa per 2 paia di scarpe di tipo B

In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [ ]:
a = np.array([2]*5)
b = np.array([1, 2, 3, 4, 5])
print(gp.quicksum(a*b))
print(a@b)

30.0
30


In [1]:
'''
uso 30 variabili, 15 per quante ne faccio ad ogni stabilimento e 15 per quantità materia prima per ogni tipo di materia prima

xA[5] quantita di scarpe A per ogni stabilimento
xB[5] quantita di scarpe B per ogni stabilimento
xC[5] quantita di scarpe C per ogni stabilimento

xP[5] quantita di blocchi di pelle per ogni stabilimento
xL[5] quantita di blocchi di lacci per ogni stabilimento
xS[5] quantita di blocchi di suole per ogni stabilimento

prezzo di ogni blocco:
pP = 4.5 * 10
pL = 0.73 * 5
pS = 7 * 3

funzione obiettivo = guadagno tot - spesa tot
guadagno tot = 30 * sum(xA) + 55 * sum(xB) + 40 * sum(xC)
spesa tot = pP * sum(xP) + pL * sum(xL) + pS * sum(xS)

vincolo sul prezzo:
pP * sum(xP) + pL * sum(xL) + pS * sum(xS) <= 1000

vincolo1:
sum(xA) >= 5

vincolo2:
xA[2] + xB[2] + xC[2] = xA[0] + xB[0] + xC[0]

vincolo3:
xB[3] >= 2

vincoli sulla produzione: (per ogni i)
la quantita richiesta di materiali non puo essere maggiore di quella che hai comprato di ogni materiale
1 * xA[i] + 0.7 * xB[i] + 1.35 * xC[i] <= 10 * xP[i] 

0.5 * xA[i] + 0.45 * xB[i] + 0.9 * xC[i] <= 5 * xL[i]

1 * xA[i] + 1 * xB[i] + 1 * xC[i] <= 3 * xS[i]

vincoli su quanto posso fare in ogni stabilimento:
(non puoi fare piu del massimo che puoi fare)
xA[0] <= 4

xA[1] <= 12

xA[2] <= 3

xA[3] <= 9

xA[4] <= 7
e avanti cosi per tutti

'''

'\nuso 30 variabili, 15 per quante ne faccio ad ogni stabilimento e 15 per quantità materia prima per ogni tipo di materia prima\n\nxA[5] quantita di scarpe A per ogni stabilimento\nxB[5] quantita di scarpe B per ogni stabilimento\nxC[5] quantita di scarpe C per ogni stabilimento\n\nxP[5] quantita di blocchi di pelle per ogni stabilimento\nxL[5] quantita di blocchi di lacci per ogni stabilimento\nxS[5] quantita di blocchi di suole per ogni stabilimento\n\nprezzo di ogni blocco:\npP = 4.5 * 10\npL = 0.73 * 5\npS = 7 * 3\n\nfunzione obiettivo = guadagno tot - spesa tot\nguadagno tot = 30 * sum(xA) + 55 * sum(xB) + 40 * sum(xC)\nspesa tot = pP * sum(xP) + pL * sum(xL) + pS * sum(xS)\n\nvincolo sul prezzo:\npP * sum(xP) + pL * sum(xL) + pS * sum(xS) <= 1000\n\nvincolo1:\nsum(xA) >= 5\n\nvincolo2:\nxA[2] + xB[2] + xC[2] = xA[0] + xB[0] + xC[0]\n\nvincolo3:\nxB[3] >= 2\n\nvincoli sulla produzione: (per ogni i)\nla quantita richiesta di materiali non puo essere maggiore di quella che hai comp

In [2]:
m = gp.Model()
nstab = 5
nmod = 3

#le variabili devono essere minori dei valori nella tabella (massimi che posso fare)
xA = m.addVars(nstab, ub=[4, 12, 3, 9, 7], vtype=GRB.INTEGER)
xB = m.addVars(nstab, ub=[2, 7, 3, 14, 5], vtype=GRB.INTEGER)
xC = m.addVars(nstab, ub=[8, 3, 5, 2, 9], vtype=GRB.INTEGER)

xP = m.addVars(nstab, vtype=GRB.INTEGER)
xL = m.addVars(nstab, vtype=GRB.INTEGER)
xS = m.addVars(nstab, vtype=GRB.INTEGER)

#prezzi dei "blocchi"
pP = 4.5 * 10
pL = 0.73 * 5
pS = 7 * 3

#vincolo sull'acquisto
m.addConstr(pP * gp.quicksum(xP) + pL * gp.quicksum(xL) + pS * gp.quicksum(xS) <= 1000)

#vincolo1
m.addConstr(gp.quicksum(xA) >= 5)

#vincolo2:
m.addConstr(xA[2] + xB[2] + xC[2] == xA[0] + xB[0] + xC[0])

#vincolo3:
m.addConstr(xB[3] >= 2)

#vincolo sui materiali
for i in range(nstab):#per ogni stabilimento
    m.addConstr(2 * xA[i] + 1.4 * xB[i] + 2.7 * xC[i] <= 10 * xP[i])#la quantità di materiali che uso dev essere minore di quella che ho comprato

    m.addConstr(1 * xA[i] + 0.9 * xB[i] + 1.8 * xC[i] <= 5 * xL[i])

    m.addConstr(2 * xA[i] + 2 * xB[i] + 2 * xC[i] <= 3 * xS[i])


guadagno_tot = 30 * gp.quicksum(xA) + 55 * gp.quicksum(xB) + 40 * gp.quicksum(xC)
spesa_tot = pP * gp.quicksum(xP) + pL * gp.quicksum(xL) + pS * gp.quicksum(xS)

m.setObjective(guadagno_tot - spesa_tot, GRB.MAXIMIZE)

m.optimize()

if m.status == GRB.OPTIMAL:
    for t in range(nstab):
        print(f"Stabilimento {t+1}:")
        print(f"  Scarpe A:  {int(xA[t].X)}")
        print(f"  Scarpe B:  {int(xB[t].X)}")
        print(f"  Scarpe C:  {int(xC[t].X)}")

        print(f"  Pelle:   {int(xP[t].X)}")
        print(f"  Lacci:   {int(xL[t].X)}")
        print(f"  Suole:   {int(xS[t].X)}")

        print('Obj: %g' % m.ObjVal)
else:
    print("Il modello non ha trovato una soluzione ottimale")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Core(TM) i5-6300U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 19 rows, 30 columns and 87 nonzeros (Max)
Model fingerprint: 0x3d28a1a1
Model has 30 linear objective coefficients
Variable types: 0 continuous, 30 integer (0 binary)
Coefficient statistics:
  Matrix range     [9e-01, 4e+01]
  Objective range  [4e+00, 6e+01]
  Bounds range     [2e+00, 1e+01]
  RHS range        [2e+00, 1e+03]

Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 18 rows, 30 columns, 86 nonzeros
Variable types: 0 continuous, 30 integer (0 binary)
Found heuristic solution: objective 76.7000000
Found heuristic solution: objective 91.7000000

Root relaxation: objective 1.192435e+03, 8 iterations, 0.00 seconds (0.00 work units)

/tmp/ipykernel_20032/2788371117.py:20: DeprecationWarning: Calling quicksum on a tupledict is deprecated, use .sum() instead.
  m.addConstr(pP * gp.quicksum(xP) + pL * gp.quicksum(xL) + pS * gp.quicksum(xS) <= 1000)
/tmp/ipykernel_20032/2788371117.py:23: DeprecationWarning: Calling quicksum on a tupledict is deprecated, use .sum() instead.
  m.addConstr(gp.quicksum(xA) >= 5)
/tmp/ipykernel_20032/2788371117.py:40: DeprecationWarning: Calling quicksum on a tupledict is deprecated, use .sum() instead.
  guadagno_tot = 30 * gp.quicksum(xA) + 55 * gp.quicksum(xB) + 40 * gp.quicksum(xC)
/tmp/ipykernel_20032/2788371117.py:41: DeprecationWarning: Calling quicksum on a tupledict is deprecated, use .sum() instead.
  spesa_tot = pP * gp.quicksum(xP) + pL * gp.quicksum(xL) + pS * gp.quicksum(xS)


     0     0 1145.84838    0   18 1071.85000 1145.84838  6.90%     -    0s
     0     2 1145.84838    0   18 1071.85000 1145.84838  6.90%     -    0s
H   17    20                    1087.2000000 1143.92148  5.22%   4.1    0s
*  249    17              11    1090.8500000 1111.82250  1.92%   4.8    0s

Cutting planes:
  Gomory: 1
  MIR: 33
  StrongCG: 7

Explored 285 nodes (1483 simplex iterations) in 0.31 seconds (0.03 work units)
Thread count was 4 (of 4 available processors)

Solution count 10: 1090.85 1087.2 1071.85 ... 546.8

Optimal solution found (tolerance 1.00e-04)
Best objective 1.090850000000e+03, best bound 1.090850000000e+03, gap 0.0000%
Stabilimento 1:
  Scarpe A:  3
  Scarpe B:  2
  Scarpe C:  0
  Pelle:   1
  Lacci:   1
  Suole:   4
Obj: 1090.85
Stabilimento 2:
  Scarpe A:  0
  Scarpe B:  7
  Scarpe C:  0
  Pelle:   1
  Lacci:   2
  Suole:   5
Obj: 1090.85
Stabilimento 3:
  Scarpe A:  0
  Scarpe B:  3
  Scarpe C:  2
  Pelle:   1
  Lacci:   2
  Suole:   4
Obj: 1090.85
Stabi